In [5]:
import os
from datasets import Dataset
from openai import OpenAI  # Using official OpenAI client instead of LangChain
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory

# Ragas metrics
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    FactualCorrectness,
    ResponseGroundedness,
    AnswerAccuracy
)
from ragas import evaluate

os.environ["OPENAI_API_KEY"] = "dummy-key"

# 1. Create an official OpenAI client pointing to your local Ollama /v1 endpoint
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="dummy-key"
)

# 2. Use Ragas's NEW factory methods (passing the Ollama client)
ragas_llm = llm_factory(
    model="llama3",
    client=ollama_client
)

ragas_emb = embedding_factory(
    provider="openai", # We tell Ragas it's "OpenAI", but we pass our Ollama client
    model="nomic-embed-text",
    client=ollama_client
)

# 3. Define the Test Dataset
data = {
    'question': ['What is the capital of France?'],
    'answer': ['Paris is the capital.'],
    'contexts' : [['France is a country in Europe. Its capital is Paris.']],
}
dataset = Dataset.from_dict(data)

# 4. Initialize metrics with the new native Ragas LLM/Embedding objects
metrics = [
    Faithfulness(llm=ragas_llm),
    AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb), # Needs both
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
    FactualCorrectness(llm=ragas_llm),
    ResponseGroundedness(llm=ragas_llm),
    AnswerAccuracy(llm=ragas_llm)
]

# 5. Run Ragas Evaluation
print("Evaluating model outputs directly via Ollama...")

score = evaluate(
    dataset=dataset,
    metrics=metrics
)

print("\n--- Evaluation Results ---")
print(score.to_pandas())

C:\Users\Sujeet\AppData\Local\Temp\ipykernel_44744\2360606081.py:33: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = embedding_factory(


Evaluating model outputs directly via Ollama...


TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]